In [1]:
!pip install sentence-transformers -q

In [ ]:
%%writefile train.py

import os
import torch
import pandas as pd
from datasets import Dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from transformers import EarlyStoppingCallback

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# 1. LOAD DATA
df = pd.read_csv('/kaggle/input/datasets/ngpham06/datafinetune-nowsgm/Final_triplet_Data_nowsgm.csv')
df = df.dropna(subset=['query', 'positive', 'hard_neg']).reset_index(drop=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df_val = df[:1000].reset_index(drop=True)
df_train = df[1000:].reset_index(drop=True)

print(f"Train: {len(df_train)} | Val: {len(df_val)}")

# 2. CHUẨN BỊ DATASET
def prepare_dataset(df):
    data = {
        'anchor': ["query: " + str(q) for q in df['query'].tolist()],
        'positive': ["passage: " + str(p) for p in df['positive'].tolist()],
        'negative': ["passage: " + str(n) for n in df['hard_neg'].tolist()]
    }
    return Dataset.from_dict(data)

train_dataset = prepare_dataset(df_train)
val_dataset = prepare_dataset(df_val)
print(train_dataset)

# 3. LOAD MODEL
model = SentenceTransformer("intfloat/multilingual-e5-base")

# 4. LOSS FUNCTION
loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=128)

# 5. EVALUATOR
val_queries = {f"q_{i}": "query: " + str(q) for i, q in enumerate(df_val['query'].tolist())}

val_corpus = {}
relevant_docs = {}
for i, row in df_val.iterrows():
    p_id = f"p_{i}"
    n_id = f"n_{i}"
    val_corpus[p_id] = "passage: " + str(row['positive'])
    val_corpus[n_id] = "passage: " + str(row['hard_neg'])
    relevant_docs[f"q_{i}"] = {p_id}

evaluator = InformationRetrievalEvaluator(
    queries=val_queries,
    corpus=val_corpus,
    relevant_docs=relevant_docs,
    name='val_eval',
)

# 6. TRAINING ARGUMENTS
args = SentenceTransformerTrainingArguments(
    output_dir='/kaggle/working/vietnamese-embedding-finetuned',

    num_train_epochs=3,
    per_device_train_batch_size=256,
    per_device_eval_batch_size=256,
    gradient_accumulation_steps=1,

    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,

    fp16=True,
    bf16=False,
    gradient_checkpointing=True,

    eval_strategy='steps',
    eval_steps=2000,
    save_strategy='steps',
    save_steps=2000,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='val_eval_cosine_ndcg@10',

    logging_steps=100,
    report_to='none',
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    dataloader_drop_last=True,
)

# 7. TRAINER
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    loss=loss,
    evaluator=evaluator,
    callbacks=[EarlyStoppingCallback(
        early_stopping_patience=2,
        early_stopping_threshold=0.001
    )]
)

# 8. TRAIN
print("Bắt đầu train...")
trainer.train()

# 9. LƯU MODEL
model.save_pretrained('/kaggle/working/vietnamese-embedding-finetuned/final')
print("Done! Model saved.")

Writing train.py


In [3]:
!torchrun --nproc_per_node=2 train.py

W0526 08:17:02.556000 49 torch/distributed/run.py:852] 
W0526 08:17:02.556000 49 torch/distributed/run.py:852] *****************************************
W0526 08:17:02.556000 49 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0526 08:17:02.556000 49 torch/distributed/run.py:852] *****************************************
Train: 251978 | Val: 1000
Train: 251978 | Val: 1000
Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 251978
})
Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 251978
})
modules.json: 100%|████████████████████████████| 387/387 [00:00<00:00, 1.46MB/s]
README.md: 179kB [00:00, 70.5MB/s]
sentence_bert_config.json: 100%|██████████████| 57.0/57.0 [00:00<00:00, 263kB/s]
config.json: 100%|█████████████████████████████| 694/694 [00:00<00:0